In [1]:
import pandas as pd
pd.options.display.max_columns = None
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, Perceptron, SGDClassifier, RidgeClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, cross_val_score, train_test_split, HalvingGridSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

### Preprocessing

In [3]:
df = pd.read_csv('fraudTrain.csv', index_col=0, parse_dates=['trans_date_trans_time'])

Based on discoveries of EDA, I delete some unusable featurs and transform others.

In [4]:
df['industry_code'] = df['cc_num'].astype(str).apply(lambda x: x[0:1])
df['age'] = (pd.to_datetime('today') - pd.to_datetime(df['dob'])).dt.days/365.25 
df.drop(columns=['first', 'last', 'street', 'trans_num', 'unix_time', 'cc_num', 'dob'], inplace=True)


In [ ]:
X_train, y_train, X_test, y_test = train_test_split(df.drop(columns='is_fraud'), df['is_fraud'], test_size=0.25, random_state=1)

In [5]:
df['trans_date_trans_time'] = df['trans_date_trans_time'].dt.second

le = LabelEncoder()
df['merchant'] = le.fit_transform(df['merchant'])
df['category'] = le.fit_transform(df['category'])
df['gender'] = le.fit_transform(df['gender'])
df['city'] = le.fit_transform(df['city'])
df['state'] = le.fit_transform(df['state'])
df['job'] = le.fit_transform(df['job'])

In [ ]:
scale before test


2. Transform
3. Test
4. Generate
5. Test
6. Select
7. Test, test learning
8. Imbalance
9. Test, test learning

### Feature generation

In [7]:
def evaluate(X, y):
    models = dict()
    
    models['Ridge Classifier'] = RidgeClassifier(random_state=42, max_iter=100000, class_weight='balanced')
    models['Logistic Regression'] = LogisticRegression(random_state=42, max_iter=100000, class_weight='balanced')
    #models['Decision Tree'] = DecisionTreeClassifier(random_state=42, class_weight='balanced')
    models['Random Forest'] = RandomForestClassifier(random_state=42, class_weight='balanced')
    models['Extra Trees'] = ExtraTreesClassifier(random_state=42, class_weight='balanced')
    models['Gradient Boosting'] = GradientBoostingClassifier(random_state=42)
    models['Hist Gradient Boosting'] = HistGradientBoostingClassifier(random_state=42)
    models['AdaBoost'] = AdaBoostClassifier(random_state=42)
    models['SGD'] = SGDClassifier(random_state=42, class_weight='balanced')
    models['SVC'] = SVC(class_weight='balanced', random_state=42)
    models['Nearest Neighbor'] = KNeighborsClassifier(3)
    models['Perceptron'] = Perceptron(random_state=42)
    models['MLPC'] = MLPClassifier(random_state=42)

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=1)
    
    names, res = [], []

    for name, model in models.items():
        print('         ', name)
        scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=10)
        res.append(scores)
        names.append(name)

    plt.figure(figsize=(10, 8))
    plt.boxplot(res, labels=names, showmeans=True)
    plt.grid()
    plt.xticks(rotation=45);

In [ ]:
evaluate(X_train, y_train)

          Ridge Classifier
          Logistic Regression
          Random Forest
          Extra Trees
          Gradient Boosting
          Hist Gradient Boosting
          AdaBoost
          SGD
          SVC


### Feature selection

### Handling imbalanced data